# Notebook 02: Class-Conditional Variational Autoencoder (C-VAE)
**Project:** Construction Site AI — Generative Safety Monitoring  
**Model Architecture:** Convolutional Conditional VAE with Gaussian Prior ($z \in \mathbb{R}^{128}$)  
**Paradigm:** Smooth Generative Latent Manifold & Controlled State Synthesis  
**Hardware:** NVIDIA GeForce RTX 3050 6GB Laptop GPU (Tier B)  

---

## 1. Mathematical Formulation & ELBO Derivation
Unlike a deterministic autoencoder, the Conditional VAE models the continuous data distribution $p(x|y)$ conditioned on worker safety class $y \in \{0, 1, 2, 3\}$.

1. **Evidence Lower Bound (ELBO):**
   $$\log p_\theta(x|y) \ge \mathbb{E}_{q_\phi(z|x, y)} [\log p_\theta(x|z, y)] - \beta \cdot D_{\text{KL}}(q_\phi(z|x, y) \parallel p(z))$$
2. **Reparameterization Trick:** Enables backpropagation through stochastic sampling:
   $$z = \mu_\phi(x, y) + \sigma_\phi(x, y) \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$
3. **Analytical KL Divergence for Gaussian Prior:**
   $$D_{\text{KL}}(\mathcal{N}(\mu, \Sigma) \parallel \mathcal{N}(0, I)) = -\frac{1}{2} \sum_{j=1}^d \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$
4. **KL Warmup Schedule:** To prevent posterior collapse early in training, $\beta$ anneals linearly over $T_{\text{warmup}} = 10$ epochs:
   $$\beta(t) = \beta_{\text{target}} \cdot \min\left(1.0, \frac{t}{T_{\text{warmup}}}\right)$$

In [ ]:
import os
import json
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Ensure working directory is project root
if not os.path.exists('configs') and os.path.exists('../configs'):
    os.chdir('..')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Project Directory: {os.getcwd()}")

## 2. Model Architecture & Latent Space Dimensions
We load the Conditional VAE weights (`runs/cvae_baseline/best.pt`).

In [ ]:
from src.models.vae import ConditionalVAE

model = ConditionalVAE(in_channels=3, latent_dim=128, num_classes=4, channels=(64, 128, 256, 512), class_emb_dim=32).to(device)
ckpt_path = 'runs/cvae_baseline/best.pt'

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded C-VAE checkpoint from Epoch {ckpt.get('epoch')} (Val Loss: {ckpt.get('val_loss'):.5f})")
else:
    print("C-VAE checkpoint not found; model ready for training.")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,} ({total_params * 4 / (1024**2):.2f} MB float32)")
print(f"Latent Space Dimension: {model.latent_dim} | Conditioning Classes: {model.num_classes}")

## 3. Training Convergence & KL Annealing Trajectory
We examine the balance between reconstruction loss and KL divergence as $\beta$ warm-up progresses.

In [ ]:
from src.common.dataset import get_crop_dataloader

test_loader = get_crop_dataloader('processed/crops/crops_test_64.npz', batch_size=128, shuffle=False, normalize_mode='sigmoid')
all_latents, all_labels = [], []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        imgs = batch['image'].to(device)
        lbls = batch['label'].to(device)
        mu, _ = model.encode(imgs, lbls)
        all_latents.append(mu.cpu())
        all_labels.append(lbls.cpu())

all_latents = torch.cat(all_latents, dim=0)
all_labels = torch.cat(all_labels, dim=0).numpy()

latents_centered = all_latents - all_latents.mean(dim=0, keepdim=True)
U, S, V = torch.pca_lowrank(latents_centered, q=2)
latents_2d = torch.matmul(latents_centered, V[:, :2]).numpy()
explained_var = (S[:2]**2) / (all_latents.shape[0] - 1)
total_var = torch.var(latents_centered, dim=0).sum()
ratio = (explained_var / total_var).numpy()

plt.figure(figsize=(9, 7))
colors = ['#2ca02c', '#d62728', '#ff7f0e', '#9467bd']
for c in range(4):
    mask = (all_labels == c)
    plt.scatter(latents_2d[mask, 0], latents_2d[mask, 1], c=colors[c], label=class_names[c], alpha=0.5, s=20)

plt.xlabel(f'PCA Component 1 ({ratio[0]*100:.1f}%)', fontsize=11)
plt.ylabel(f'PCA Component 2 ({ratio[1]*100:.1f}%)', fontsize=11)
plt.title('2D Latent Manifold Projection (PyTorch PCA)', fontsize=12, fontweight='bold')
plt.legend(markerscale=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Class-Conditioned Generation
We sample latent vectors $z \sim \mathcal{N}(0, I)$ and decode them conditioned on each safety class:
- Class 0: Compliant (Wearing hardhat and vest)
- Class 1: Missing Helmet
- Class 2: Missing Vest
- Class 3: Both Missing

In [ ]:
class_names = ['Compliant (0)', 'No Helmet (1)', 'No Vest (2)', 'Both Missing (3)']
model.eval()

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
with torch.no_grad():
    for c in range(4):
        z = torch.randn(4, model.latent_dim, device=device)
        labels = torch.full((4,), c, dtype=torch.long, device=device)
        samples = model.decode(z, labels).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
        
        for col in range(4):
            axes[c, col].imshow(samples[col])
            axes[c, col].axis('off')
            if col == 0:
                axes[c, col].set_title(class_names[c], fontsize=11, fontweight='bold', loc='left')

plt.suptitle('C-VAE Conditional Worker Generation by Safety Class', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 5. Smooth Latent Space Interpolations
We demonstrate that the learned latent space is continuous by linearly interpolating between two random latent coordinates $z_1, z_2$:
$$z(\alpha) = (1 - \alpha) z_1 + \alpha z_2, \quad \alpha \in [0, 1]$$

In [ ]:
steps = 10
alphas = torch.linspace(0.0, 1.0, steps, device=device).unsqueeze(1)
z1 = torch.randn(1, model.latent_dim, device=device)
z2 = torch.randn(1, model.latent_dim, device=device)
z_interp = (1.0 - alphas) * z1 + alphas * z2

fig, axes = plt.subplots(4, steps, figsize=(16, 6))
with torch.no_grad():
    for c in range(4):
        labels = torch.full((steps,), c, dtype=torch.long, device=device)
        imgs = model.decode(z_interp, labels).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
        for s in range(steps):
            axes[c, s].imshow(imgs[s])
            axes[c, s].axis('off')
            if s == 0:
                axes[c, s].set_ylabel(class_names[c], fontsize=10, fontweight='bold')

plt.suptitle('Smooth Latent Space Interpolation Trajectory Across Safety States', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Latent Space 2D Manifold Visualization
We project latent representations of real test crops into 2D using Principal Component Analysis (PCA) to confirm that classes form structured, continuous clusters.

In [ ]:
from src.common.dataset import get_crop_dataloader
from sklearn.decomposition import PCA

test_loader = get_crop_dataloader('processed/crops/crops_test_64.npz', batch_size=128, shuffle=False, normalize_mode='sigmoid')
all_latents = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        imgs = batch['image'].to(device)
        lbls = batch['label'].to(device)
        mu, _ = model.encode(imgs, lbls)
        all_latents.append(mu.cpu().numpy())
        all_labels.append(lbls.cpu().numpy())

all_latents = np.concatenate(all_latents, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

pca = PCA(n_components=2)
latents_2d = pca.fit_transform(all_latents)

plt.figure(figsize=(9, 7))
colors = ['#2ca02c', '#d62728', '#ff7f0e', '#9467bd']
for c in range(4):
    mask = (all_labels == c)
    plt.scatter(latents_2d[mask, 0], latents_2d[mask, 1], c=colors[c], label=class_names[c], alpha=0.5, s=20)

plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
plt.title('2D Latent Manifold Projection (PCA)', fontsize=12, fontweight='bold')
plt.legend(markerscale=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Conclusions & Findings
- **Smooth Manifold:** Conditioning on class labels successfully steers generator output toward specific violation states while preserving continuous transitions between worker poses.
- **Trade-off with Deterministic AE:** The Gaussian prior regularization produces smoother interpolations and valid random sampling, but results in slightly softer edge definition compared to the deterministic autoencoder.